# LIBRARY

In [ ]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, make_scorer, roc_auc_score
import shap

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import statsmodels.api as sm

# LOAD

In [6]:
with open("PROCESSED/DATA/merged_and_dropped.cat_cols.json") as f:
    cat_cols = json.load(f)

X_train = pd.read_parquet("INPUTS/TRAIN/X_train.parquet")
X_test = pd.read_parquet("INPUTS/TEST/X_test.parquet")
y_train = pd.read_parquet("INPUTS/TRAIN/y_train.parquet")
y_test = pd.read_parquet("INPUTS/TEST/y_test.parquet")

X_train[cat_cols] = X_train[cat_cols].astype("category")
X_test[cat_cols] = X_test[cat_cols].astype("category")

num_cols = [c for c in X_train.columns if c not in cat_cols]

# one-hot encode categorical variables
X_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# ensure target is categorical
y_train_cat = y_train.iloc[:, 0].astype("category")
y_test_cat  = y_test.iloc[:, 0].astype("category")

# BUILD PIPELINE
- `optimize_model`: Utilizes GridSearchCV for hyperparameter optimization
- `evaluate_best_model`: Evaluates the best model on train and test data
- `save_results`: Saves results and computes and saves SHAP values

In [ ]:
# CONFIGURATION
cv_random_state = 42
base_path = Path("RESULTS/BASELINES")
baseline_probs_path = base_path / "PROBABILITIES"
baseline_SHAP_path = base_path / "SHAP"
baseline_performance_path = base_path / "PERFORMANCE"
baseline_params_path = base_path / "PARAMETERS"

def optimize_model(model, X, y, param_grid, verbose = True, save_results = True):
    scorers = {
        'auc': 'roc_auc_ovr',
        'accuracy': make_scorer(accuracy_score),
        'f1': make_scorer(f1_score, average='macro')
    }
    refit = 'f1'

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=cv_random_state)

    grid = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = scorers,
        refit = refit,
        cv = cv,
        n_jobs = -1,
        verbose = 1
    )
    grid.fit(X, y)
    
    if verbose:
        print("Best parameters:", grid.best_params_)
        print(f"Best {refit} CV score:", grid.best_score_)
        print("CV AUC at best-AUC params:", grid.cv_results_['mean_test_auc'][grid.best_index_])
        print("CV F1 at best-AUC params:", grid.cv_results_['mean_test_f1'][grid.best_index_])
        print("CV Accuracy at best-AUC params:", grid.cv_results_['mean_test_accuracy'][grid.best_index_])

    return grid

def evaluate_best_model(model, X_train, y_train, X_test, y_test, save_results = True):
    best_model = model.best_estimator_

    y_pred_train = best_model.predict(X_train)
    y_pred_test  = best_model.predict(X_test)

    acc_train = accuracy_score(y_train, y_pred_train)
    acc_test  = accuracy_score(y_test, y_pred_test)

    f1_train = f1_score(y_train, y_pred_train, average='macro')
    f1_test  = f1_score(y_test, y_pred_test, average='macro')

    auc_train = roc_auc_score(y_train, best_model.predict_proba(X_train)[:, 1])
    auc_test  = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

    print(f"Train accuracy: {acc_train:.3f},  F1: {f1_train:.3f}, AUC: {auc_train:.3f}")
    print(f"Test  accuracy: {acc_test:.3f},  F1: {f1_test:.3f}, AUC: {auc_test:.3f}")

    print("\nClassification report:\n")
    cls_report = classification_report(y_test, y_pred_test)
    print(cls_report)
    return {
        'acc_train': acc_train, 
        'acc_test': acc_test,
        'f1_train': f1_train,
        'f1_test': f1_test,
        'auc_train': auc_train,
        'auc_test': auc_test,
        'cls_report': cls_report
    }

# Saves results + computes and saves SHAP values
def save_results(model, X_train, y_train, X_test, y_test, performance_dict):
    best_model = model.best_estimator_
    best_params = model.best_params_
    cv_results = model.cv_results_
    best_index = model.best_index_

    model_name = best_model.__class__.__name__
    os.makedirs(baseline_probs_path, exist_ok=True)
    os.makedirs(baseline_SHAP_path, exist_ok=True)
    os.makedirs(baseline_performance_path, exist_ok=True)
    os.makedirs(baseline_params_path, exist_ok=True)

    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_train) # Compute shap values

    # SAVE PARAMETERS
    with open(baseline_params_path / f"{model_name}_BEST_HYPER.json", "w") as f:
        json.dump(best_params, f)

    # SAVE PROBABILITY OUTPUTS
    test_df = pd.DataFrame(best_model.predict_proba(X_test))
    test_df["y_true"] = np.asarray(y_test)
    test_df.to_csv(baseline_probs_path / f"{model_name}_PROBS.csv", index=False)

    # SAVE SHAP IMPORTANCES
    if shap_values.ndim == 3: # "Multiclass" case (some models treat binary as multiclass)
        mean_abs_shap_class = np.abs(shap_values).mean(axis=(0,2))
    else: # binary case
        mean_abs_shap_class = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame(mean_abs_shap_class, index=X_train.columns, columns=["mean_abs_shap"])
    shap_df.to_csv(baseline_SHAP_path / f"{model_name}_SHAP.csv", index=True)
    
    # SAVE PERFORMANCE
    performance = {
        "model": model_name,
        "acc_train": performance_dict['acc_train'],
        "acc_test": performance_dict['acc_test'],
        "f1_train": performance_dict['f1_train'],
        "f1_test": performance_dict['f1_test'],
        "auc_train": performance_dict['auc_train'],
        "auc_test": performance_dict['auc_test'],
        "best_cv_auc": cv_results['mean_test_auc'][best_index],
        "best_cv_f1": cv_results['mean_test_f1'][best_index],
        "best_cv_accuracy": cv_results['mean_test_accuracy'][best_index]
    }
    performance = pd.DataFrame([performance])
    performance.to_csv(baseline_performance_path / f"{model_name}_PERF.csv", index=False)
    
    with open(baseline_performance_path / f"{model_name}_CLASSIFICATION_REPORT.txt", "w") as f:
        f.write(performance_dict['cls_report'])

# MODELS

### RANDOM FOREST

In [9]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

model = optimize_model(rf, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best f1 CV score: 0.6055491814248094
CV AUC at best-AUC params: 0.8748648921872636
CV F1 at best-AUC params: 0.6055491814248094
CV Accuracy at best-AUC params: 0.8512009333198058
Train accuracy: 0.944,  F1: 0.887, AUC: 0.997
Test  accuracy: 0.873,  F1: 0.649, AUC: 0.897

Classification report:

              precision    recall  f1-score   support

         0.0       0.87      0.99      0.93      1642
         1.0       0.85      0.24      0.37       306

    accuracy                           0.87      1948
   macro avg       0.86      0.61      0.65      1948
weighted avg       0.87      0.87      0.84      1948



### XGB

In [11]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

model = optimize_model(xgb, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best f1 CV score: 0.7756154122564114
CV AUC at best-AUC params: 0.9170421794955876
CV F1 at best-AUC params: 0.7756154122564114
CV Accuracy at best-AUC params: 0.8892035256990257
Train accuracy: 0.944,  F1: 0.893, AUC: 0.982
Test  accuracy: 0.918,  F1: 0.825, AUC: 0.937

Classification report:

              precision    recall  f1-score   support

         0.0       0.93      0.98      0.95      1642
         1.0       0.83      0.60      0.70       306

    accuracy                           0.92      1948
   macro avg       0.88      0.79      0.82      1948
weighted avg       0.91      0.92      0.91      1948



### EXTRA TREES

In [12]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

et = ExtraTreesClassifier(random_state=42, n_jobs=-1)

model = optimize_model(et, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best f1 CV score: 0.5104545616107697
CV AUC at best-AUC params: 0.8550139240416597
CV F1 at best-AUC params: 0.5104545616107697
CV Accuracy at best-AUC params: 0.8350238284109178
Train accuracy: 0.908,  F1: 0.792, AUC: 0.987
Test  accuracy: 0.851,  F1: 0.515, AUC: 0.857

Classification report:

              precision    recall  f1-score   support

         0.0       0.85      1.00      0.92      1642
         1.0       0.90      0.06      0.11       306

    accuracy                           0.85      1948
   macro avg       0.88      0.53      0.51      1948
weighted avg       0.86      0.85      0.79      1948



### CATBOOST

In [13]:
param_grid = {
    'n_estimators': [200, 400],
    'depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
}

cat = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    thread_count=-1,
    verbose=0  # silence per-iteration output
)

model = optimize_model(cat, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters: {'depth': 6, 'learning_rate': 0.1, 'n_estimators': 400}
Best f1 CV score: 0.7703019760507166
CV AUC at best-AUC params: 0.9176223628729633
CV F1 at best-AUC params: 0.7703019760507166
CV Accuracy at best-AUC params: 0.8884326646901456
Train accuracy: 0.994,  F1: 0.990, AUC: 1.000
Test  accuracy: 0.916,  F1: 0.823, AUC: 0.937

Classification report:

              precision    recall  f1-score   support

         0.0       0.93      0.97      0.95      1642
         1.0       0.81      0.60      0.69       306

    accuracy                           0.92      1948
   macro avg       0.87      0.79      0.82      1948
weighted avg       0.91      0.92      0.91      1948



### HIST GRADIENT BOOSTING

In [14]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'max_iter': [100, 200],
    'min_samples_leaf': [20, 50],
}

hgb = HistGradientBoostingClassifier(
    random_state=42
)

model = optimize_model(hgb, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 200, 'min_samples_leaf': 50}
Best f1 CV score: 0.7746419147862257
CV AUC at best-AUC params: 0.9141030007416472
CV F1 at best-AUC params: 0.7746419147862257
CV Accuracy at best-AUC params: 0.8889462259305559
Train accuracy: 0.945,  F1: 0.894, AUC: 0.981
Test  accuracy: 0.911,  F1: 0.811, AUC: 0.934

Classification report:

              precision    recall  f1-score   support

         0.0       0.93      0.97      0.95      1642
         1.0       0.79      0.59      0.67       306

    accuracy                           0.91      1948
   macro avg       0.86      0.78      0.81      1948
weighted avg       0.91      0.91      0.91      1948



### LIGHTGBM

In [8]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 63],
}

lgbm = LGBMClassifier(
    objective='binary',
    random_state=42,
    n_jobs=-1,
    verbose=-1      # <- this suppresses the info messages
)

model = optimize_model(lgbm, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'num_leaves': 63}
Best f1 CV score: 0.7726669112287533
CV AUC at best-AUC params: 0.9113779088995339
CV F1 at best-AUC params: 0.7726669112287533
CV Accuracy at best-AUC params: 0.8899734472967604
Train accuracy: 0.999,  F1: 0.998, AUC: 1.000
Test  accuracy: 0.918,  F1: 0.827, AUC: 0.938

Classification report:

              precision    recall  f1-score   support

         0.0       0.93      0.97      0.95      1642
         1.0       0.81      0.62      0.70       306

    accuracy                           0.92      1948
   macro avg       0.87      0.80      0.83      1948
weighted avg       0.91      0.92      0.91      1948



c:\Users\homel\miniconda3\envs\brigand\Lib\site-packages\shap\explainers\_tree.py:587: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


### UNTUNED GLM

Special Handling for the baseline GLM model, since it uses statsmodels instead of sklearn + is not a tree model

In [ ]:
X_train_glm_untuned = sm.add_constant(X_encoded.astype(float))
X_test_glm_untuned = sm.add_constant(X_test_encoded.astype(float))

glm_untuned = sm.GLM(
    y_train,
    X_train_glm_untuned,
    family = sm.families.Binomial()
).fit()

# Predictions
y_pred_train_glm_untuned = glm_untuned.predict(X_train_glm_untuned)
y_pred_test_glm_untuned = glm_untuned.predict(X_test_glm_untuned)

y_pred_train_cat_untuned = (y_pred_train_glm_untuned >= 0.5).astype(int)
y_pred_test_cat_untuned = (y_pred_test_glm_untuned >= 0.5).astype(int)
acc_train_untuned = accuracy_score(y_train_cat, y_pred_train_cat_untuned)
acc_test_untuned = accuracy_score(y_test_cat, y_pred_test_cat_untuned)
f1_train_untuned = f1_score(y_train_cat, y_pred_train_cat_untuned, average='macro')
f1_test_untuned = f1_score(y_test_cat, y_pred_test_cat_untuned, average='macro')
auc_train_untuned = roc_auc_score(y_train_cat, y_pred_train_glm_untuned)
auc_test_untuned = roc_auc_score(y_test_cat, y_pred_test_glm_untuned)

print(f"Train accuracy: {acc_train_untuned:.3f},  F1: {f1_train_untuned:.3f}, AUC: {auc_train_untuned:.3f}")
print(f"Test  accuracy: {acc_test_untuned:.3f},  F1: {f1_test_untuned:.3f}, AUC: {auc_test_untuned:.3f}")
print("\nClassification report:\n")
cls_report = classification_report(y_test_cat, y_pred_test_cat_untuned)
print(cls_report)

# Probabilty Outputs
test_df_untuned = pd.DataFrame(y_pred_test_glm_untuned, columns=["prob_1"])
test_df_untuned["y_true"] = np.asarray(y_test)
test_df_untuned.to_csv(baseline_probs_path / "GLM_UNTUNED_PROBS.csv", index=False)

# Performance metrics
model_name = "GLM_UNTUNED"
perf_row = {
    "model": model_name,
    "acc_train": acc_train_untuned,
    "acc_test": acc_test_untuned,
    "f1_train": f1_train_untuned,
    "f1_test": f1_test_untuned,
    "auc_train": auc_train_untuned,
    "auc_test": auc_test_untuned,
}
perf_df = pd.DataFrame([perf_row])
perf_df.to_csv(baseline_performance_path / f"{model_name}_PERF.csv", index=False)

with open(baseline_performance_path / f"{model_name}_CLASSIFICATION_REPORT.txt", "w") as f:
    f.write(cls_report)

print(glm_untuned.summary())

c:\Users\homel\miniconda3\envs\brigand\Lib\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)


Train accuracy: 0.903,  F1: 0.788, AUC: 0.738
Test  accuracy: 0.913,  F1: 0.803, AUC: 0.757

Classification report:

              precision    recall  f1-score   support

         0.0       0.92      0.98      0.95      1642
         1.0       0.86      0.53      0.66       306

    accuracy                           0.91      1948
   macro avg       0.89      0.76      0.80      1948
weighted avg       0.91      0.91      0.90      1948

                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7418
Model Family:                Binomial   Df Model:                          370
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                    nan
Date:                Mon, 24 Nov 2025   Deviance:                       69906.
Tim

c:\Users\homel\miniconda3\envs\brigand\Lib\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
c:\Users\homel\miniconda3\envs\brigand\Lib\site-packages\statsmodels\genmod\families\family.py:1056: RuntimeWarning: divide by zero encountered in log
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +
c:\Users\homel\miniconda3\envs\brigand\Lib\site-packages\statsmodels\genmod\families\family.py:1056: RuntimeWarning: invalid value encountered in multiply
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +
